# Comparing and Selecting LLMs for Enterprise Processes

**Estimated duration:** 90-120 minutes

This complete reference module compares one **baseline** logical model with one deliberate **change**. It uses the repository's real provider contracts and deterministic offline fixtures, so every learner can run the demonstrations without credentials or billable requests. Fixture latency, token, and price values are labelled `simulated_offline_fixture`; they are teaching evidence, not provider benchmarks. Use the separate [workshop copy](workshops/15_compare_and_select_llms_exercises.ipynb) when you want to implement the four exercises yourself.

## Module Guide

1. Route two logical models through one frozen golden dataset.
2. Automate blinded, order-balanced side-by-side judging and calculate win rates.
3. Compare output throughput, session token economics, cost coverage, and monthly TCO.
4. Fail deployment eligibility when approved governance or liability evidence is missing or non-compliant.

## Learning Objectives

By the end of the module, you will be able to:

1. Build an apples-to-apples A/B accuracy test that fixes cases, prompts, parameters, and scoring rules while changing only the logical model.
2. Compute LLM-as-a-Judge wins, losses, ties, decisive win rates, and position-bias indicators without exposing model identity to the judge.
3. Estimate output tokens per second, cost per multi-turn session, monthly TCO, and missing-cost coverage without treating unknown cost as zero.
4. Apply automated, fail-closed compliance checks for retention, data use, residency, network, audit, enterprise terms, and liability evidence before any deployment or inference request.
5. Record the result with the platform lifecycle vocabulary: `baseline -> change -> result -> decision`, where the decision is `adopt`, `reject`, or `inconclusive`.


## Introduction & Setup

Run the notebook from the repository workspace. The locked contributor path remains `make examples-install`; the next cell provides the requested Jupyter `%pip install` equivalent and installs the repository's existing extras without adding a dependency. Restart the kernel if Jupyter reports that imported packages changed.

Authentication is deliberately keyless. Connected models are selected by logical name from `aai-platform.yml` and authenticate through the configured Azure identity. If an enterprise gateway also requires a key, put only a `keyvault://...` or `databricks-secret://...` reference in configuration or the optional environment variable below. Never paste, display, log, or commit a raw key.


In [ ]:
import sys
from pathlib import Path

# Jupyter form: %pip install -e "<repo>[databricks,genai,examples]"
repo_root = next(
    (
        directory
        for directory in (Path.cwd(), *Path.cwd().parents)
        if (directory / "pyproject.toml").is_file()
        and (directory / "examples" / "notebook_setup.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open the cloned repository as your workspace.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
get_ipython().run_line_magic(
    "pip",
    f'install -e "{repo_root}[databricks,genai,examples]"',
)


In [ ]:
import os

from aai_core import bootstrap
from aai_core.testing import dev_context

RUN_CONNECTED_MODELS = (
    os.getenv("AAI_RUN_CONNECTED_MODEL_COMPARISON", "false").casefold() == "true"
)
MODEL_LOGICAL_NAMES = tuple(
    filter(
        None,
        (
            name.strip()
            for name in os.getenv(
                "AAI_MODEL_COMPARISON_LOGICAL_NAMES", "baseline-chat,change-chat"
            ).split(",")
        ),
    )
)
if len(MODEL_LOGICAL_NAMES) != 2:
    raise ValueError("Configure exactly two logical model names.")
optional_secret_reference = os.getenv("AAI_MODEL_OPTIONAL_SECRET_REFERENCE")
if optional_secret_reference and not optional_secret_reference.startswith(
    ("keyvault://", "databricks-secret://")
):
    raise ValueError("Configure an approved secret reference, not a raw secret.")
platform_context = bootstrap() if RUN_CONNECTED_MODELS else dev_context()
optional_secret = (
    platform_context.secrets.resolve(optional_secret_reference)
    if RUN_CONNECTED_MODELS and optional_secret_reference
    else None
)
print(
    {
        "mode": "connected" if RUN_CONNECTED_MODELS else "offline_fixture",
        "logical_models": MODEL_LOGICAL_NAMES,
        "authentication": "configured keyless identity",
        "optional_secret": (
            "resolved and redacted" if optional_secret else "not required"
        ),
    }
)

## 1. Routing 2 Models through a Golden Dataset (Apples-to-Apples Accuracy)

A fair model comparison freezes every material input except the model: the ordered case set, prompt, temperature, maximum output tokens, scoring rules, and repetition policy. Use logical resource names so configuration—not notebook code—maps the same experiment to approved endpoints in each environment. Preserve a canonical dataset digest and row-level results; an average alone can hide a critical failure.

| Strategy | Best use case | Weak point |
|---|---|---|
| Exact-match or required-term checks | Structured extraction, routing labels, citations, policy phrases | Misses semantically correct paraphrases |
| Same-case baseline/change A/B | Explaining whether a model change caused the result | Invalid if prompts, parameters, or cases also change |
| Repeated seeded or bounded trials | Measuring stochastic stability | More requests, cost, and analysis complexity |
| Aggregate accuracy plus critical-row gates | Release decisions where rare errors have high impact | Requires domain owners to identify critical cases |

**Practical Application.** A finance operations team evaluates invoice exception routing. Both logical models receive the same missing-PO, duplicate-invoice, and tax-mismatch cases at temperature zero. The change improves average wording but incorrectly approves the missing-PO case; the row-level critical gate rejects the change even if its overall accuracy looks acceptable.

The fixture below runs through `PlatformContext.providers.model(...)`, the same stable adapter boundary used by connected endpoints. Replace the injected fixture models with two configured logical names to make real calls; the comparison function does not change.


In [ ]:
from examples.support.model_selection import (
    GOLDEN_CASES,
    golden_fixture_context,
    run_golden_ab,
)

fixture_context = golden_fixture_context()
golden_report = run_golden_ab(
    fixture_context, ("baseline-chat", "change-chat"), GOLDEN_CASES
)
golden_report["summary"]

### Now you try - exercise

Review the completed `run_golden_comparison()` reference below, or implement it independently in the [workshop copy](workshops/15_compare_and_select_llms_exercises.ipynb). Its return value contains `rows` and `summary`; each summary entry contains `accuracy`.

1. Resolve both logical models through `context.providers.model(...)` and send every case to both with `temperature=0.0`.
2. Mark a row as passed only when every `required_term` appears, case-insensitively, in the response.
3. Calculate per-model accuracy and preserve the same ordered case IDs so reviewers can audit paired failures.


In [ ]:
from examples.support.model_selection import run_golden_comparison

In [ ]:
# Verification -- do not modify
from aai_core.testing import FakeChatModel, dev_context

exercise_context = dev_context()
for name, reply in (("baseline-chat", "ESCALATE safely"), ("change-chat", "APPROVE")):
    exercise_context.providers.register_model(
        name, FakeChatModel(logical_name=name, reply=reply)
    )
exercise_cases = [
    {
        "inputs": {
            "case_id": "missing-document",
            "request": "A required document is missing.",
        },
        "expectations": {"required_terms": ("ESCALATE",)},
    }
]
exercise_report = run_golden_comparison(
    exercise_context, ("baseline-chat", "change-chat"), exercise_cases
)
assert exercise_report["summary"]["baseline-chat"]["accuracy"] == 1.0
assert exercise_report["summary"]["change-chat"]["accuracy"] == 0.0
assert [row["case_id"] for row in exercise_report["rows"]] == ["missing-document"] * 2
print("Verification passed: both models used the same golden case.")

## 2. Automating Side-by-Side Scoring with LLM-as-a-Judge (Win Rates)

Pairwise judging asks an approved judge model which of two responses better satisfies a versioned rubric. Blind the response identities, swap A/B presentation order, retain ties, and map the judge's label back to the logical model only after scoring. A win rate is not ground truth: deterministic rules run first, judge cost is reported separately, and the judge stays report-only until it agrees sufficiently with held-out human labels. This module defines **decisive win rate** as wins divided by wins plus losses; ties are reported separately.

| Strategy | Best use case | Weak point |
|---|---|---|
| Single blinded pairwise pass | Fast directional comparison of nuanced responses | Vulnerable to position bias and stochasticity |
| Swapped-order pairwise passes | Detecting whether A/B placement changes the verdict | Doubles judge requests and cost |
| Pointwise rubric scores | Absolute thresholding against one standard | Scores may be less discriminating than direct comparison |
| Human-calibrated judge | Scalable semantic scoring after validation | Requires reviewed labels, versioning, and ongoing drift checks |

**Practical Application.** A claims team compares two denial explanations. The change is clearer but sometimes omits the policy clause. A blinded judge prefers its tone, while the deterministic citation rule fails it. The platform records the judge preference but rejects the change because semantic preference cannot override a mandatory control.


In [ ]:
from examples.support.model_selection import (
    PAIRWISE_CASES,
    pairwise_judge_fixture,
    run_balanced_pairwise_judge,
)

judge_report = run_balanced_pairwise_judge(
    pairwise_judge_fixture(), PAIRWISE_CASES, ("baseline-chat", "change-chat")
)
judge_report["summary"]

### Now you try - exercise

Review the completed `summarize_pairwise_verdicts()` reference below, or implement it in the [workshop copy](workshops/15_compare_and_select_llms_exercises.ipynb). It returns `models`, `tie_rate`, `position_A_win_rate`, and `position_B_win_rate`; each model entry contains `wins`, `losses`, and `decisive_win_rate`.

1. Map each blinded `winner_label` back through `model_A` or `model_B`; do not infer model identity from answer text.
2. Keep ties outside the decisive denominator and report their rate separately.
3. Calculate A- and B-position win rates so a reviewer can detect label-order bias.


In [ ]:
from examples.support.model_selection import summarize_pairwise_verdicts

In [ ]:
# Verification -- do not modify
from examples.support.model_selection import SAMPLE_VERDICTS

pairwise_result = summarize_pairwise_verdicts(
    SAMPLE_VERDICTS, ("baseline-chat", "change-chat")
)
assert pairwise_result["models"]["change-chat"]["wins"] == 2
assert pairwise_result["models"]["baseline-chat"]["losses"] == 2
assert pairwise_result["models"]["change-chat"]["decisive_win_rate"] == 1.0
assert pairwise_result["tie_rate"] == 0.5
assert pairwise_result["position_A_win_rate"] == 0.25
assert pairwise_result["position_B_win_rate"] == 0.25
print("Verification passed: logical wins are invariant to A/B position.")

## 3. Evaluating TCO & Token Economics (Comparing TPS and Session Token Costs)

Enterprise TCO is a workload calculation, not a comparison of advertised per-token prices. Measure input and output tokens separately, define throughput explicitly (here, output tokens per second), aggregate complete multi-turn sessions, and include expected monthly session volume. Use trace or gateway/billing evidence for real chargeback. Prices passed into this fixture are deliberately simulated; never embed changeable vendor prices in application code. Quality, safety, and governance eligibility are hard filters before any cost ranking.

| Strategy | Best use case | Weak point |
|---|---|---|
| Per-token request cost | Comparing similar prompts at small scale | Hides retries, judge calls, caching, and long sessions |
| Cost per complete session | Forecasting an enterprise business process | Requires representative turn counts and context growth |
| Output tokens per second | User-visible generation throughput | Does not include queueing or time to first token |
| Gateway or billing chargeback | Authoritative production allocation | Often arrives later and needs workload attribution tags |

**Practical Application.** A service desk change is cheaper per million tokens but produces longer answers and repeats more context on every turn. Session-level modeling reveals that it costs more per resolved ticket. The team keeps the baseline until the change either shortens sessions or demonstrates enough quality improvement to justify the higher TCO.


In [ ]:
from examples.support.model_selection import (
    SIMULATED_APPROVED_PRICE_CARD,
    SIMULATED_SESSION_OBSERVATIONS,
    compare_session_economics,
)

economic_report = compare_session_economics(
    SIMULATED_SESSION_OBSERVATIONS, SIMULATED_APPROVED_PRICE_CARD, 50_000
)
eligible_economics = [row for row in economic_report if row["cost_comparable"]]
illustrative_lowest_tco = min(
    eligible_economics, key=lambda row: row["monthly_tco_usd"]
)["logical_model"]
{
    "report": economic_report,
    "illustrative_lowest_tco": illustrative_lowest_tco,
    "decision": "inconclusive",
    "reason": "simulated values are not release evidence",
}

### Now you try - exercise

Review the completed `estimate_session_economics()` reference below, or implement it in the [workshop copy](workshops/15_compare_and_select_llms_exercises.ipynb). It returns `output_tokens_per_second`, `session_cost_usd`, `monthly_tco_usd`, and `cost_coverage`.

1. Calculate output TPS as output tokens divided by latency seconds; reject a non-positive latency.
2. Calculate input and output cost independently from per-million-token rates, then scale one representative session to the monthly volume.
3. If either price is unknown, return `None` for both costs and `0.0` for coverage—never substitute zero cost.


In [ ]:
from examples.support.model_selection import estimate_session_economics

In [ ]:
# Verification -- do not modify
sample_usage = {"input_tokens": 1000, "output_tokens": 500, "latency_ms": 2000}
sample_price = {"input_usd_per_million": 2.0, "output_usd_per_million": 6.0}
economics = estimate_session_economics(sample_usage, sample_price, 10_000)
assert economics["output_tokens_per_second"] == 250.0
assert abs(economics["session_cost_usd"] - 0.005) < 1e-12
assert abs(economics["monthly_tco_usd"] - 50.0) < 1e-9
assert economics["cost_coverage"] == 1.0
unknown = estimate_session_economics(
    sample_usage,
    {"input_usd_per_million": None, "output_usd_per_million": 6.0},
    10_000,
)
assert unknown["session_cost_usd"] is None
assert unknown["monthly_tco_usd"] is None
assert unknown["cost_coverage"] == 0.0
print("Verification passed: TPS and complete/unknown cost behave correctly.")

## 4. Enterprise Governance & Liability Checks (Before Deployment)

Governance is an eligibility gate, not another quality score. The application must consume versioned, approved evidence from the enterprise provider catalog and fail closed before resolving or calling a model. The SDK does not administer or live-inspect vendor retention, contracts, or network controls, and a `data_classification` tag alone does not enforce them. The offline record below demonstrates evidence shape and reuses `aai_core.evaluation.apply_gate()`; it does not claim that a provider control was verified live.

| Strategy | Best use case | Weak point |
|---|---|---|
| Contract and DPA evidence | Retention, training use, indemnity, subprocessors | Can become stale unless versioned and re-attested |
| Security-control evidence | Private connectivity, encryption, audit logging | A declaration is weaker than tested platform telemetry |
| Data-residency allowlist | Regulated regional processing | Region names alone do not prove every subprocessor path |
| Automated fail-closed preflight | Blocking an ineligible model before requests or deployment | Requires an authoritative external evidence owner |

**Practical Application.** A newly available change model looks cheaper and wins the semantic comparison, but its approved record says zero data retention is disabled and content may be retained for 30 days. The preflight fails `zero_data_retention` and `retention_days`, so the model is automatically rejected before any enterprise data is sent.


In [ ]:
from examples.support.model_selection import (
    governance_fixtures,
    run_governance_preflight,
)

baseline_governance, change_governance = governance_fixtures()
governance_report = [
    run_governance_preflight(evidence, required_residency="canada")
    for evidence in (baseline_governance, change_governance)
]
governance_report

### Now you try - exercise

Review the completed `gate_model_governance()` reference below, or implement it in the [workshop copy](workshops/15_compare_and_select_llms_exercises.ipynb). It returns `approved` and a `failures` list of stable control names; missing controls fail closed.

1. Require zero retention, zero retention days, disabled customer-data training, the required residency, keyless authentication, controlled networking, audit logging, approved enterprise terms, and approved liability terms.
2. Add a failure for every missing or non-compliant control rather than stopping at the first one.
3. Return only evidence and decisions—never call or deploy the model from this function.


In [ ]:
from examples.support.model_selection import gate_model_governance

In [ ]:
# Verification -- do not modify
from examples.support.model_selection import UNSAFE_GOVERNANCE_EVIDENCE

governance_result = gate_model_governance(
    UNSAFE_GOVERNANCE_EVIDENCE, required_residency="canada"
)
assert governance_result["approved"] is False
assert {
    "zero_data_retention",
    "retention_days",
    "private_network_or_controlled_egress",
    "liability_terms_approved",
}.issubset(set(governance_result["failures"]))
assert "required_residency" not in governance_result["failures"]
print("Verification passed: the unsafe change is blocked before inference.")

## Production Handoff

You now have four distinct evidence layers: deterministic golden-case accuracy, calibrated semantic preference, workload economics, and governance eligibility. Do not collapse them into one weighted score. Governance, safety, critical-row quality, and minimum cost coverage are independent gates; TCO ranks only models that pass them.

The offline worksheet's final decision is **`inconclusive`** and its release is **blocked until connected evaluation**. For a production comparison:

1. Configure two logical model names in environment-specific `aai-platform.yml`; keep physical endpoint and deployment names out of application code.
2. Persist the reviewed golden records as a versioned Unity Catalog EvaluationDataset and bind both runs to its exact version or digest.
3. Trace real calls with an approved capture policy, record target and judge costs separately, and source chargeback from the gateway or billing system.
4. Read governance evidence from the approved external provider catalog, re-attest it on a schedule, and run the preflight before resolving or invoking a model.
5. Record the evidence as `baseline -> change -> result -> decision`; only an explicit `adopt` decision may name an immutable eligible release.
